# AIC354 - Machine Learning Fundamentals
## Lab 06: K-Nearest Neighbor (KNN) Algorithm
### Complete Solved Activities & Graded Lab Tasks Notebook

---
- **Student Name:** Muhammad Afaq
- **Registration Number:** FA23-BSE-012
- **Degree Program:** BS Software Engineering (BSE) - Section A
- **Course:** AIC354 - Machine Learning Fundamentals
- **Institution:** COMSATS University Islamabad

---
### Notebook Outline:
1. **Theoretical Concepts:** Intuition, Distance Metrics, Hyperparameter $K$, Feature Scaling
2. **Solved Activity 1:** Iris Flower Classification using KNN ($k=5$)
3. **Solved Activity 2:** Breast Cancer Classification in 2D Feature Space with Scatter Plots & Decision Boundary
4. **Graded Lab Task 1:** Handwritten Digits (0-9) Classification with Hyperparameter $K$ Optimization (Manual & `GridSearchCV`), Confusion Matrix & Classification Report
5. **Graded Lab Task 2:** Sea Slug (Abalone) Age Prediction via KNN Regression with Physical Measurements (EDA, Normalization, Distance-weighted KNN, Residuals)


---
## 0. Environment Setup & Core Imports


In [ ]:
import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets
from sklearn.datasets import load_breast_cancer, load_digits
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score
)

# Styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'
print('Libraries loaded successfully!')


---
# Part 1: Solved Lab Activity 1
## Iris Dataset Classification using KNN Algorithm ($k=5$)

**Objective:**
Classify 3 species of Iris flowers (*Setosa, Versicolor, Virginica*) using 4 morphometric features:
1. Sepal Length (cm)
2. Sepal Width (cm)
3. Petal Length (cm)
4. Petal Width (cm)



In [ ]:
# 1. Load Iris dataset
iris = datasets.load_iris()
X_iris = iris.data
y_iris = iris.target
target_names_iris = iris.target_names

print(f"Iris Dataset shape: {X_iris.shape[0]} samples, {X_iris.shape[1]} features")
print(f"Classes: {list(target_names_iris)}")

# 2. Train-Test Split (70% Train, 30% Test as defined in the Lab Manual)
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris
)
print(f"Train samples: {len(X_train_iris)} | Test samples: {len(X_test_iris)}")

# 3. Fit KNeighborsClassifier (n_neighbors=5)
clf_iris = KNeighborsClassifier(n_neighbors=5)
clf_iris.fit(X_train_iris, y_train_iris)

# 4. Predictions on test data
y_pred_iris = clf_iris.predict(X_test_iris)

# 5. Evaluation: Confusion Matrix & Classification Report
cm_iris = confusion_matrix(y_test_iris, y_pred_iris)
acc_iris = accuracy_score(y_test_iris, y_pred_iris)

print(f"Test Accuracy: {acc_iris * 100:.2f}%
")
print("Confusion Matrix:")
print(cm_iris)
print("
Classification Report:")
print(classification_report(y_test_iris, y_pred_iris, target_names=target_names_iris))


In [ ]:
# Visualize Iris Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm_iris, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names_iris, yticklabels=target_names_iris,
            annot_kws={"size": 13, "weight": "bold"})
plt.title(f"Activity 1: Iris KNN Confusion Matrix (k=5)\nAccuracy: {acc_iris*100:.2f}%", fontsize=12, fontweight='bold')
plt.xlabel("Predicted Species", fontweight='bold')
plt.ylabel("Actual Species", fontweight='bold')
plt.tight_layout()
plt.show()


---
# Part 2: Solved Lab Activity 2
## Breast Cancer Classification in 2D Feature Space

**Objective:**
Classify breast tumors as either **Malignant (0)** or **Benign (1)** using a 2D feature subspace:
- `mean area`
- `mean compactness`

Evaluate classification accuracy, render true vs predicted scatter comparisons, and visualize the non-linear decision boundary generated by KNN.



In [ ]:
# 1. Load Breast Cancer dataset
cancer = load_breast_cancer()
df_cancer_feat = pd.DataFrame(cancer.data, columns=cancer.feature_names)

# Select two features as specified in manual
X_cancer = df_cancer_feat[['mean area', 'mean compactness']].copy()
y_cancer_cat = pd.Categorical.from_codes(cancer.target, cancer.target_names)
y_cancer_dummies = pd.get_dummies(y_cancer_cat, drop_first=True) # 1 = benign, 0 = malignant
y_cancer = y_cancer_dummies.iloc[:, 0].astype(int)

print(f"Dataset Shape: {X_cancer.shape[0]} samples, {X_cancer.shape[1]} features")
print("Target distribution:")
print(y_cancer.value_counts())

# 2. Train-test split (25% test, random_state=1 as in manual)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_cancer, y_cancer, test_size=0.25, random_state=1
)
print(f"Train samples: {len(X_train_c)} | Test samples: {len(X_test_c)}")

# 3. Train KNN Classifier (k=5, metric='euclidean')
knn_cancer = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn_cancer.fit(X_train_c, y_train_c)

# 4. Predictions & Evaluation
y_pred_c = knn_cancer.predict(X_test_c)
cm_c = confusion_matrix(y_test_c, y_pred_c)
acc_c = accuracy_score(y_test_c, y_pred_c)
correct_c = cm_c[0, 0] + cm_c[1, 1]
total_c = len(y_test_c)

print(f"Confusion Matrix:\n{cm_c}")
print(f"\nGiven our confusion matrix, our model has an accuracy of {correct_c}/{total_c} = {acc_c*100:.1f}%.")
print("\nClassification Report:")
print(classification_report(y_test_c, y_pred_c, target_names=['malignant', 'benign']))


In [ ]:
# Visualizing Ground Truth vs KNN Predicted Labels
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

scatter_df = X_test_c.copy()
scatter_df['True Class'] = ['Benign (1)' if v == 1 else 'Malignant (0)' for v in y_test_c]
scatter_df['Pred Class'] = ['Benign (1)' if v == 1 else 'Malignant (0)' for v in y_pred_c]

sns.scatterplot(
    x='mean area', y='mean compactness', hue='True Class',
    palette={'Benign (1)': '#2B6CB0', 'Malignant (0)': '#E53E3E'},
    data=scatter_df, ax=axes[0], s=60, alpha=0.85, edgecolor='k'
)
axes[0].set_title("Ground Truth Test Samples", fontsize=12, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.5)

sns.scatterplot(
    x='mean area', y='mean compactness', hue='Pred Class',
    palette={'Benign (1)': '#3182CE', 'Malignant (0)': '#DD6B20'},
    data=scatter_df, ax=axes[1], s=60, alpha=0.85, edgecolor='k'
)
axes[1].set_title(f"KNN Predicted Labels (Accuracy: {acc_c*100:.1f}%)", fontsize=12, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
# Visualizing KNN Decision Boundary in 2D Feature Space
plt.figure(figsize=(8, 6))
x_min, x_max = X_cancer['mean area'].min() - 50, X_cancer['mean area'].max() + 50
y_min, y_max = X_cancer['mean compactness'].min() - 0.05, X_cancer['mean compactness'].max() + 0.05
xx, yy = np.meshgrid(np.arange(x_min, x_max, 5.0), np.arange(y_min, y_max, 0.005))
Z = knn_cancer.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.25, cmap=plt.cm.coolwarm)
scatter = plt.scatter(X_test_c['mean area'], X_test_c['mean compactness'],
                      c=y_test_c, cmap=plt.cm.coolwarm, edgecolors='k', s=55, alpha=0.85)
plt.title(f"Activity 2: KNN Decision Boundary (k=5)\nAccuracy: {acc_c*100:.1f}%", fontsize=12, fontweight='bold')
plt.xlabel("Mean Area", fontweight='bold')
plt.ylabel("Mean Compactness", fontweight='bold')
plt.colorbar(scatter, ticks=[0, 1], label='Class (0: Malignant, 1: Benign)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


---
# Part 3: Graded Lab Task 1
## Handwritten Digits (0 to 9) Classification using KNN

**Requirements from Lab Manual:**
1. Load `digits` dataset from `sklearn.datasets` (1797 samples, 64 features corresponding to 8x8 pixel grayscale values).
2. Classify digits (0 to 9) using KNN classifier. Try different values of $K$ neighbors manually and/or with `GridSearchCV` to find the value of $K$ that maximizes score.
3. Plot Confusion Matrix.
4. Plot Classification Report.



In [ ]:
# 1. Load Digits dataset
digits = load_digits()
X_digits = digits.data
y_digits = digits.target
img_digits = digits.images

print(f"Digits samples: {X_digits.shape[0]} | Features per image: {X_digits.shape[1]} (8x8 pixels)")
print(f"Unique classes: {np.unique(y_digits)}")

# 2. Train-test split (75% train, 25% test, stratified)
X_train_d, X_test_d, y_train_d, y_test_d, img_train_d, img_test_d = train_test_split(
    X_digits, y_digits, img_digits, test_size=0.25, random_state=42, stratify=y_digits
)
print(f"Train samples: {len(X_train_d)} | Test samples: {len(X_test_d)}")


In [ ]:
# 3. K Exploration: Evaluating K from 1 to 25
k_range = list(range(1, 26))
train_scores_d = []
test_scores_d = []
cv_scores_d = []

for k in k_range:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train_d, y_train_d)
    train_scores_d.append(knn_k.score(X_train_d, y_train_d))
    test_scores_d.append(knn_k.score(X_test_d, y_test_d))
    cv_scores_d.append(cross_val_score(knn_k, X_train_d, y_train_d, cv=5).mean())

best_manual_k = k_range[int(np.argmax(cv_scores_d))]
print(f"Best manual K from 5-Fold Cross Validation: k={best_manual_k} (CV Accuracy: {max(cv_scores_d)*100:.2f}%)")


In [ ]:
# 4. Hyperparameter Tuning with GridSearchCV
param_grid = {
    'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 20],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}
grid_search_digits = GridSearchCV(
    KNeighborsClassifier(), param_grid, cv=5, scoring='accuracy', n_jobs=-1
)
grid_search_digits.fit(X_train_d, y_train_d)

best_knn_digits = grid_search_digits.best_estimator_
print(f"Optimal Hyperparameters: {grid_search_digits.best_params_}")
print(f"Best 5-Fold CV Accuracy: {grid_search_digits.best_score_ * 100:.2f}%")

# 5. Evaluate on Unseen Test Data
y_pred_d = best_knn_digits.predict(X_test_d)
acc_d = accuracy_score(y_test_d, y_pred_d)
cm_d = confusion_matrix(y_test_d, y_pred_d)
cr_d_dict = classification_report(y_test_d, y_pred_d, output_dict=True)

print(f"Test Accuracy: {acc_d * 100:.2f}% ({np.sum(y_pred_d == y_test_d)}/{len(y_test_d)} correct)\n")
print("Classification Report:")
print(classification_report(y_test_d, y_pred_d, digits=4))


In [ ]:
# Visualization 1: Accuracy vs K Neighbors
plt.figure(figsize=(9, 4.8))
plt.plot(k_range, [s * 100 for s in train_scores_d], 'o--', color='#E53E3E', label='Training Accuracy', alpha=0.7)
plt.plot(k_range, [s * 100 for s in test_scores_d], 's-', color='#3182CE', label='Test Accuracy', linewidth=2)
plt.plot(k_range, [s * 100 for s in cv_scores_d], '^-', color='#2F855A', label='5-Fold CV Accuracy', linewidth=2)
plt.axvline(grid_search_digits.best_params_['n_neighbors'], color='#D69E2E', linestyle=':', linewidth=2, label=f"Optimal k={grid_search_digits.best_params_['n_neighbors']}")
plt.title("Task 1: Digits KNN Accuracy vs K Neighbors", fontsize=12, fontweight='bold')
plt.xlabel("Number of Neighbors (k)", fontweight='bold')
plt.ylabel("Accuracy (%)", fontweight='bold')
plt.xticks(k_range)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 2: Digits Confusion Matrix Heatmap
plt.figure(figsize=(8, 6.5))
sns.heatmap(cm_d, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10),
            annot_kws={"size": 11, "weight": "bold"})
plt.title(f"Task 1: Digits Confusion Matrix (Accuracy: {acc_d*100:.2f}%)\nOptimal KNN: k={grid_search_digits.best_params_['n_neighbors']}", fontsize=12, fontweight='bold')
plt.xlabel("Predicted Digit", fontweight='bold')
plt.ylabel("Actual Digit", fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 3: Per-Digit Classification Report Metrics
digits_lbls = [str(i) for i in range(10)]
prec_vals = [cr_d_dict[str(i)]['precision'] * 100 for i in range(10)]
rec_vals = [cr_d_dict[str(i)]['recall'] * 100 for i in range(10)]
f1_vals = [cr_d_dict[str(i)]['f1-score'] * 100 for i in range(10)]

x_ind = np.arange(10)
bw = 0.26

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.bar(x_ind - bw, prec_vals, bw, label='Precision (%)', color='#2B6CB0')
ax.bar(x_ind, rec_vals, bw, label='Recall (%)', color='#38A169')
ax.bar(x_ind + bw, f1_vals, bw, label='F1-Score (%)', color='#DD6B20')

ax.set_title("Task 1: Per-Digit Classification Metrics (Precision, Recall, F1-Score)", fontsize=12, fontweight='bold')
ax.set_xlabel("Digit Class (0 - 9)", fontweight='bold')
ax.set_ylabel("Score (%)", fontweight='bold')
ax.set_xticks(x_ind)
ax.set_xticklabels(digits_lbls, fontweight='bold')
ax.set_ylim(90, 102)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend(frameon=True, facecolor='white', framealpha=0.9, loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# Visualization 4: Sample Predictions Display
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for idx, ax in enumerate(axes.flat):
    img = img_test_d[idx]
    actual = y_test_d[idx]
    pred = y_pred_d[idx]
    ax.imshow(img, cmap=plt.cm.gray_r, interpolation='nearest')
    color = 'green' if actual == pred else 'red'
    ax.set_title(f"True: {actual} | Pred: {pred}", color=color, fontsize=10, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("Task 1: Sample Test Handwritten Digits & KNN Predictions", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


---
# Part 4: Graded Lab Task 2
## Use kNN to Predict the Age of Sea Slugs (Abalone Dataset)

**Background & Task Objectives:**
- Dataset: UCI Machine Learning Repository (4,177 samples, 9 attributes).
- Biological context: The age of an abalone is determined by cutting its shell, staining it, and counting the rings under a microscope ($Age = Rings + 1.5$ years). This destructive process is labor-intensive and damages specimen value.
- Machine Learning Goal: Build a non-invasive KNN regression model to accurately predict age based solely on non-destructive physical measurements (Length, Diameter, Height, Whole weight, Shucked weight, Viscera weight, Shell weight, and Sex).



In [ ]:
# 1. Load Abalone dataset
col_names_abalone = [
    'Sex', 'Length', 'Diameter', 'Height',
    'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Rings'
]
df_abalone = pd.read_csv('abalone.data', names=col_names_abalone)
df_abalone['Age'] = df_abalone['Rings'] + 1.5

print(f"Abalone Dataset shape: {df_abalone.shape[0]} rows, {df_abalone.shape[1]} columns")
print(f"Age range: {df_abalone['Age'].min():.1f} to {df_abalone['Age'].max():.1f} years (Mean: {df_abalone['Age'].mean():.2f})")
df_abalone.head(5)


In [ ]:
# 2. Correlation Analysis
num_cols = ['Length', 'Diameter', 'Height', 'Whole_weight', 'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Age']
corr_abalone = df_abalone[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_abalone, annot=True, fmt='.2f', cmap='coolwarm', vmin=-0.2, vmax=1.0, linewidths=0.5)
plt.title("Task 2: Abalone Physical Features Correlation Heatmap", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 3. Preprocessing: Encoding and Scaling
df_ab_encoded = pd.get_dummies(df_abalone, columns=['Sex'], drop_first=True, dtype=float)
feat_cols = [c for c in df_ab_encoded.columns if c not in ['Rings', 'Age']]
X_ab = df_ab_encoded[feat_cols]
y_ab = df_ab_encoded['Age']

X_train_ab, X_test_ab, y_train_ab, y_test_ab = train_test_split(
    X_ab, y_ab, test_size=0.20, random_state=42
)

# Standardize features (essential for distance calculation)
scaler_ab = StandardScaler()
X_train_ab_scaled = scaler_ab.fit_transform(X_train_ab)
X_test_ab_scaled = scaler_ab.transform(X_test_ab)

print(f"Features: {feat_cols}")
print(f"Train samples: {len(X_train_ab)} | Test samples: {len(X_test_ab)}")


In [ ]:
# 4. Hyperparameter Exploration: K from 1 to 40 (Uniform vs Distance Weighting)
k_ab_vals = list(range(1, 41))
rmse_u_list = []
rmse_d_list = []

for k in k_ab_vals:
    # Uniform
    knn_u = KNeighborsRegressor(n_neighbors=k, weights='uniform')
    knn_u.fit(X_train_ab_scaled, y_train_ab)
    rmse_u_list.append(np.sqrt(mean_squared_error(y_test_ab, knn_u.predict(X_test_ab_scaled))))
    
    # Distance
    knn_d = KNeighborsRegressor(n_neighbors=k, weights='distance')
    knn_d.fit(X_train_ab_scaled, y_train_ab)
    rmse_d_list.append(np.sqrt(mean_squared_error(y_test_ab, knn_d.predict(X_test_ab_scaled))))

best_k_d = k_ab_vals[int(np.argmin(rmse_d_list))]
min_rmse_d = min(rmse_d_list)

print(f"Optimal K with distance weighting: k={best_k_d} with RMSE={min_rmse_d:.4f} years")


In [ ]:
# Visualization: K vs RMSE
plt.figure(figsize=(9, 4.8))
plt.plot(k_ab_vals, rmse_u_list, 'o-', color='#3182CE', label='Uniform Weights', linewidth=2)
plt.plot(k_ab_vals, rmse_d_list, 's-', color='#DD6B20', label='Distance Weights (1/distance)', linewidth=2)
plt.axvline(best_k_d, color='#E53E3E', linestyle='--', label=f'Optimal Distance k={best_k_d}')
plt.title("Task 2: KNN Regressor Performance - RMSE vs K Neighbors", fontsize=12, fontweight='bold')
plt.xlabel("Number of Neighbors (k)", fontweight='bold')
plt.ylabel("RMSE (Years)", fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()


In [ ]:
# 5. Final Model Evaluation
final_knn_ab = KNeighborsRegressor(n_neighbors=best_k_d, weights='distance')
final_knn_ab.fit(X_train_ab_scaled, y_train_ab)
y_pred_ab = final_knn_ab.predict(X_test_ab_scaled)

mae_ab = mean_absolute_error(y_test_ab, y_pred_ab)
mse_ab = mean_squared_error(y_test_ab, y_pred_ab)
rmse_ab = np.sqrt(mse_ab)
r2_ab = r2_score(y_test_ab, y_pred_ab)
base_rmse_ab = np.sqrt(mean_squared_error(y_test_ab, np.full_like(y_test_ab, y_train_ab.mean())))

print(f"Mean Absolute Error (MAE):       {mae_ab:.4f} years")
print(f"Root Mean Squared Error (RMSE):  {rmse_ab:.4f} years")
print(f"R² Score:                        {r2_ab:.4f} ({r2_ab*100:.2f}% variance explained)")
print(f"Baseline (Mean Predictor) RMSE:  {base_rmse_ab:.4f} years")
print(f"Error Reduction over Baseline:   {((base_rmse_ab - rmse_ab)/base_rmse_ab)*100:.2f}%")


In [ ]:
# Visualizing Actual vs Predicted Age
plt.figure(figsize=(7.5, 6))
plt.scatter(y_test_ab, y_pred_ab, alpha=0.45, color='#2B6CB0', s=35, label='Test Abalones')
min_val = min(y_test_ab.min(), y_pred_ab.min()) - 1
max_val = max(y_test_ab.max(), y_pred_ab.max()) + 1
plt.plot([min_val, max_val], [min_val, max_val], color='#E53E3E', linestyle='--', linewidth=2, label='Perfect Fit (y = x)')
plt.title(f"Task 2: Abalone Age Prediction - Actual vs Predicted\n(Optimal KNN: k={best_k_d}, RMSE={rmse_ab:.2f} yrs)", fontsize=12, fontweight='bold')
plt.xlabel("Actual Age (Years = Rings + 1.5)", fontweight='bold')
plt.ylabel("KNN Predicted Age (Years)", fontweight='bold')
plt.xlim(min_val, max_val)
plt.ylim(min_val, max_val)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True, facecolor='white', framealpha=0.9, loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# Sample Predictions Table Demonstration
sample_indices = [5, 12, 42, 100, 250]
sample_features = X_test_ab.iloc[sample_indices]
sample_actual_age = y_test_ab.iloc[sample_indices].values
sample_preds = final_knn_ab.predict(X_test_ab_scaled[sample_indices])

demo_rows = []
for i in range(len(sample_indices)):
    demo_rows.append({
        'Sample': i + 1,
        'Length (mm)': round(sample_features['Length'].iloc[i], 3),
        'Diameter (mm)': round(sample_features['Diameter'].iloc[i], 3),
        'Shell Wt (g)': round(sample_features['Shell_weight'].iloc[i], 3),
        'Actual Rings': int(sample_actual_age[i] - 1.5),
        'Actual Age (Yrs)': round(sample_actual_age[i], 1),
        'Predicted Age (Yrs)': round(sample_preds[i], 2),
        'Abs Error (Yrs)': round(abs(sample_actual_age[i] - sample_preds[i]), 2)
    })

pd.DataFrame(demo_rows)
